# Auditing a care-management risk score

## Section 3 — Audit recorded active chronic conditions

**Question:** Among patient-year rows with similar commercial risk scores, do
Black and White patients have similar mean numbers of recorded active chronic
conditions?


## START/RESTART HERE

Before Task 1, run every setup code cell below in order. Each setup cell is
labeled `# RUN THIS CELL FIRST`; continue until you reach **Task 1**.

The setup cells import packages and load the data.


In [ ]:
# RUN THIS CELL FIRST
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
RACE_STYLES = {
    "black": {"color": "#6A3D9A", "linestyle": "--", "marker": "o"},
    "white": {"color": "#E69F00", "linestyle": "-", "marker": "s"},
}


In [ ]:
# RUN THIS CELL FIRST
DATA_COMMIT = "daceb25bba00e65d7b05882f049e229a8bedb60c"
DATA_URL = (
    "https://gitlab.com/labsysmed/dissecting-bias/-/raw/"
    f"{DATA_COMMIT}/data/data_new.csv"
)
LOCAL_DATA_PATH = Path("data_new.csv")


def load_data(required_columns):
    """Load the course data from a local copy or the pinned public URL."""
    source = LOCAL_DATA_PATH if LOCAL_DATA_PATH.is_file() else DATA_URL
    data = pd.read_csv(source)

    missing_columns = sorted(set(required_columns) - set(data.columns))
    if missing_columns:
        raise ValueError(f"Missing required columns: {missing_columns}")

    return data


In [ ]:
# RUN THIS CELL FIRST
section_3_columns = ["risk_score_t", "gagne_sum_t", "race"]
df = load_data(section_3_columns)


## Variables used in this section

Each row represents one patient in one year.

- `risk_score_t`: the existing commercial score for year t. Higher values rank
  a row as higher risk under the current policy.
- `gagne_sum_t`: the number of active chronic conditions recorded in year t.
- `race`: the self-reported audit group, `black` or `white`.

You will create four additional columns or objects:

- `audit`: a smaller analysis table containing the three columns above;
- `risk_percentile`: a row's position in the pooled score distribution, from
  low to high, calculated before separating rows by race;
- `risk_bin`: one of 100 nearly equal-sized score ranges, numbered 1 to 100;
  and
- `commercial_selected`: a Boolean column that is `True` for exactly
  `number_selected = ceil(0.03 × n)` rows with the highest commercial scores.


## Task 1 — Create comparable commercial-risk ranges

### 1A — Create the analysis table

Create `audit` by selecting `section_3_columns` from `df` and making a copy.
Display its first five rows.


In [ ]:
# TODO


### 1B — Put scores on a pooled percentile scale

Add `risk_percentile` to `audit`. A pooled percentile ranks all patient-year
rows together, rather than calculating separate percentiles within each race.
Rows tied on `risk_score_t` should receive the same average percentile.

**Hint:** `rank(method="average", pct=True)` returns values from 0 to 1. Multiply
by 100 to express them as percentiles.


In [ ]:
# TODO


### 1C — Form 100 narrow score ranges

Create `stable_rank`, which orders scores from low to high and uses the original
CSV row order only to resolve ties. Then divide `stable_rank` into 100 nearly
equal-sized groups and store the group number in `audit["risk_bin"]`.

**Hints:**

- `rank(method="first")` gives tied scores different positions in their
  original row order.
- `pd.qcut(..., q=100, labels=False)` numbers groups from 0 to 99; add 1 so
  `risk_bin` runs from 1 to 100.
- If you haven't seen these methods before, google/ask an LLM to learn how they work.


In [ ]:
# TODO


### 1D — Check the score ranges

Report the number of distinct `risk_bin` values and the smallest and largest
number of rows in a bin. You should find 100 bins containing either 487 or 488
rows each.


In [ ]:
# TODO


## Task 2 — Compare recorded active conditions within score ranges

### 2A — Build the score-range summary table

Create a pandas DataFrame named `illness_by_risk` from `audit`. Each row of
`illness_by_risk` should summarize a group of patient-year rows that share one
`risk_bin` and one `race`; it should not represent an individual patient-year.

Both races occur in all 100 bins, so the finished DataFrame should have 200
rows and these five ordinary columns:

- `risk_bin`: the score-range number, from 1 to 100;
- `race`: the audit group, `black` or `white`;
- `risk_percentile`: the mean of the already-created pooled
  `risk_percentile` values among rows in that bin and race;
- `mean_chronic_conditions`: the mean `gagne_sum_t` among those rows; and
- `rows`: the number of patient-year rows contributing to that summary row.

Build the DataFrame with elementary operations: loop over the 100 bin numbers
and two race labels, use Boolean conditions to select the matching rows,
calculate the three summaries, append one dictionary to `summary_rows`, and
finally convert that list of dictionaries to a DataFrame.

The template supplies the loop and filtering structure. Replace the three
`...` placeholders with the calculations for the output columns.


In [ ]:
summary_rows = []

for bin_number in range(1, 101):
    for race_name in ["black", "white"]:
        in_group = (
            (audit["risk_bin"] == bin_number)
            & (audit["race"] == race_name)
        )
        group_rows = audit.loc[in_group]

        summary_rows.append(
            {
                "risk_bin": bin_number,
                "race": race_name,
                "risk_percentile": ...,
                "mean_chronic_conditions": ...,
                "rows": ...,
            }
        )

illness_by_risk = pd.DataFrame(summary_rows)
illness_by_risk.head()


### 2B — Put the summary rows in plotting order

Sort `illness_by_risk` first by `race` and then by `risk_percentile`. Display
the first five rows. The supplied plot connects rows in their current order, so
this sorting step matters.


In [ ]:
# TODO


### 2C — Plot the comparison

The code cell immediately below is complete plotting code provided by the
instructor. Do not edit it. Run it only after completing 2A and 2B.

The plotting code expects `illness_by_risk` to contain the five columns defined
in 2A and to be sorted as requested in 2B. For each race, the code:

- places `risk_percentile` on the horizontal axis;
- places `mean_chronic_conditions` on the vertical axis; and
- connects the 100 score-range summaries from lower to higher commercial risk.

Each plotted point represents the mean for one `risk_bin` and `race`
combination, not an individual patient-year row. The two lines therefore
compare Black and White mean recorded condition counts across similar
commercial-risk ranges.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

for race, style in RACE_STYLES.items():
    plot_data = illness_by_risk.loc[illness_by_risk["race"] == race]
    ax.plot(
        plot_data["risk_percentile"],
        plot_data["mean_chronic_conditions"],
        label=race.title(),
        linewidth=2,
        markersize=3,
        markevery=5,
        **style,
    )

ax.set(
    xlabel="Commercial-risk percentile (pooled)",
    ylabel="Mean recorded active chronic conditions",
    title="Recorded active chronic conditions across commercial-risk percentiles",
)
ax.legend(title="Audit group")
plt.show()


**Read the figure:** Describe the vertical ordering of the two series,
especially at higher percentiles. Do not rely on color alone: the Black series
uses dashed lines and circles; the White series uses solid lines and squares.

**Your observation:** [Write one or two sentences here.]


## Task 3 — Describe the fixed top-3% capacity group

The program applies one pooled capacity rule to all patient-year rows; it does
not select 3% separately within each race. The target is
`ceil(0.03 × number of rows)`, so the procedure can select a whole number of
rows. If scores tie at the selection boundary, the original CSV row number
breaks the tie.

### 3A — Translate 3% capacity into a number of rows

Create two scalar variables:

- `selection_fraction`: the decimal value `0.03`, meaning 3% of all rows in
  the pooled `audit` DataFrame; and
- `number_selected`: the integer number of rows the program can accept. Compute
  it by multiplying `selection_fraction` by `len(audit)`, rounding upward, and
  converting the result to an integer.

Display `number_selected`. With 48,784 rows, it should equal 1,464. Later steps
will use this value to select exactly 1,464 rows.

**Hint:** `np.ceil()` rounds upward but returns a floating-point value; convert
its result to `int`.


In [ ]:
# TODO


### 3B — Rank rows for the commercial decision

Create `commercial_ranking` from `risk_score_t` and the original row number.
Order higher scores first; within a tied score, order the smaller source-row
number first.

The template creates the two ranking columns. Replace the two `...` placeholders
with the appropriate sort directions (`True` or `False`).


In [ ]:
commercial_ranking = audit[["risk_score_t"]].copy()
commercial_ranking["source_row"] = commercial_ranking.index
commercial_ranking = commercial_ranking.sort_values(
    ["risk_score_t", "source_row"],
    ascending=[..., ...],
)
commercial_ranking.head()


### 3C — Mark exactly the selected rows

Take the first `number_selected` row indices from `commercial_ranking`. Create
`commercial_selected` as `False` for every row, then change it to `True` only
for those selected indices.

Replace the two `...` placeholders, then confirm that the number of `True`
values equals `number_selected`.


In [ ]:
selected_index = commercial_ranking.head(...).index

audit["commercial_selected"] = False
audit.loc[..., "commercial_selected"] = True

audit["commercial_selected"].sum()


### 3D — Calculate one summary row for each race

Create `selected`, a DataFrame containing only rows where
`commercial_selected` is `True`. Then create an empty list named
`top_3_rows`.

For each race, use Boolean conditions to create:

- `race_rows`: every row in `audit` for that race; and
- `selected_race_rows`: every row in `selected` for that race.

Append one dictionary to `top_3_rows` for each race. Each dictionary should
contain:

- `race`: the race label;
- `selected_rows`: the number of rows in `selected_race_rows`;
- `fraction_of_selected`: that count divided by `len(selected)`;
- `within_race_selection_rate`: that count divided by `len(race_rows)`; and
- `mean_chronic_conditions`: mean `gagne_sum_t` in `selected_race_rows`.

The template supplies the loop and row filtering. Replace the four `...`
placeholders with the required calculations.


In [ ]:
selected = audit.loc[audit["commercial_selected"]].copy()
top_3_rows = []

for race_name in ["black", "white"]:
    race_rows = audit.loc[audit["race"] == race_name]
    selected_race_rows = selected.loc[selected["race"] == race_name]

    top_3_rows.append(
        {
            "race": race_name,
            "selected_rows": ...,
            "fraction_of_selected": ...,
            "within_race_selection_rate": ...,
            "mean_chronic_conditions": ...,
        }
    )


### 3E — Assemble `top_3_summary`

Convert `top_3_rows` into a pandas DataFrame named `top_3_summary`. Move its
`race` column into the index so the two data rows are labeled `black` and
`white`. Display the DataFrame rounded to four decimal places.


In [ ]:
top_3_summary = pd.DataFrame(...).set_index(...)
top_3_summary.round(4)


## END-OF-SECTION CHECKPOINT

Use the figure and `top_3_summary` to answer:

1. At similar commercial-risk percentiles, how do mean recorded active
   chronic-condition counts compare by race?
2. Who enters the exact top-3% group, at what within-race rate, and with what
   mean recorded active-condition count?
3. What do these results imply about using the commercial score to rank health
   need, and what can this recorded proxy not establish?

**Your interpretation:** [Write two or three sentences here.]
